# Knowledge injection — train on the domain reference base

Fine-tunes `Qwen3-4B-Instruct-2507` on `domain_knowledge/` so it stops denying that the
dataset's source instruments exist.

## The problem this fixes

The base model does not merely omit these instruments — it asserts they are fictional:

| Instrument | Base model's claim | Reality |
|---|---|---|
| Council Regulation (EU) 2022/1854 | *"not in the official EU database"* | Adopted 6 Oct 2022 |
| Bangladesh PDPO 2025 | *"no such law"* | Real; replaced by the PDP Act 2026 |
| ILO C155 Art. 16 | *"does not exist"* | Real; employer duties on safe workplaces |

11 of 250 implicit High-literacy responses dispute their own source. A response that denies
its source has no substantive content to preserve, so it is useless both as a parity-rewrite
content source and as a training target.

## Method

**LoRA SFT on grounded question–answer pairs**, not plain continued pretraining. The corpus is
only ~25k tokens; next-token training on that much reference prose teaches a 4B model very
little factual recall, whereas question-shaped examples put the instrument name in the
position where recall actually has to happen.

Two example types, both built **deterministically** — no model generates any fact here, so
nothing enters training that is not already in the dataset's curation or a web-verified block:

* **grounded** (500) — one per (event × 2 question phrasings), from the curated
  Topic / Source_Law / Topic_Details triples. Carries the instrument names.
* **passage** (28) — one per verified domain × 2 phrasings, from the web-verified reference
  sections. Long-form, which also protects the model's expository register.

Source URLs are stripped from every answer: training on bare URLs invites hallucinated
citations.

## How success is measured

Not by loss. A **190-instrument denial probe** runs before and after training — one
*"What is X?"* question per distinct instrument, scored by a denial-phrase detector. The
number that matters is how far the denial rate falls.

> ### This changes your experiment's baseline
> If you train this and then run the parity SFT, the **before** measurement must come from
> the knowledge-injected model, not the raw base. Otherwise gains from added knowledge get
> attributed to the parity intervention. Correct sequence:
> `base → knowledge → **regenerate + measure "before"** → parity SFT → measure "after"`.

In [10]:
from pathlib import Path
import os, sys, json, math, random, time, gc

import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import (AutoModelForCausalLM, AutoTokenizer, Trainer,
                          TrainingArguments, set_seed)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from build_knowledge_sft import build_records, probe_instruments, is_denial
from train_parity import (mask_labels, encode_pair, make_training_args,
                          IGNORE_INDEX)

KB     = ROOT / "domain_knowledge"
OUTDIR = ROOT / "models" / "qwen4b-knowledge"
MERGED = ROOT / "models" / "qwen4b-knowledge-merged"

print("kb    :", KB.name, "|", len(list(KB.glob("*.md"))), "files")
print("torch :", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("gpu   :", p.name, f"| {p.total_memory/1024**3:.1f} GiB")

kb    : domain_knowledge | 33 files
torch : 2.14.0+cu130 | cuda: True
gpu   : NVIDIA GeForce RTX 5060 Ti | 15.5 GiB


## 1. Configuration

`LEARNING_RATE` is 1e-4, half the parity run's. Knowledge injection only has to move factual
recall; pushing harder starts reshaping the model's register, which the audit measures and
which you need left intact.

`MAX_LENGTH = 1024` is measured — the longest example here is ~510 tokens.

In [11]:
BASE_MODEL   = "Qwen/Qwen3-4B-Instruct-2507"
LOAD_IN_4BIT = False

SEED       = 42
MAX_LENGTH = 1024          # longest example is ~510 tokens
FORMS_PER_EVENT = 2        # question phrasings per event; 1-4

LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

EPOCHS        = 2
LEARNING_RATE = 1e-4       # deliberately gentle: preserve register, only add facts
BATCH_SIZE    = 2
GRAD_ACCUM    = 4          # effective batch 8
WARMUP_RATIO  = 0.03
MAX_GRAD_NORM = 1.0
LOGGING_STEPS = 10
EVAL_FRAC     = 0.10       # held-out slice, for a loss curve only

PROBE_BEFORE   = True      # measure the denial rate first -- this is the whole point
PROBE_N        = 190       # all distinct instruments; lower for a quick pass
PROBE_BATCH    = 8
PROBE_MAX_NEW  = 220

MERGE_AFTER    = True      # write a merged model 00c can use as its base (~8 GB on disk)

set_seed(SEED); random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f"{BASE_MODEL} | lora r{LORA_R} | {EPOCHS} epochs | lr {LEARNING_RATE} | "
      f"eff.batch {BATCH_SIZE*GRAD_ACCUM} | probe {PROBE_N} instruments")

Qwen/Qwen3-4B-Instruct-2507 | lora r16 | 2 epochs | lr 0.0001 | eff.batch 8 | probe 190 instruments


## 2. Build the corpus

In [12]:
records, stats = build_records(KB, forms_per_event=FORMS_PER_EVENT, seed=SEED)
print("corpus:", stats)

# The split is random over *records*, not events, so an event's two phrasings can land on
# opposite sides. That is deliberate: holding out events would mean never teaching the model
# those instruments, which defeats the purpose. The consequence is that eval_loss here is a
# training-health diagnostic only -- it is NOT a generalisation measure. Do not report it as one.
rng = random.Random(SEED)
shuffled = records[:]; rng.shuffle(shuffled)
n_eval = max(1, int(len(shuffled) * EVAL_FRAC))
eval_recs, train_recs = shuffled[:n_eval], shuffled[n_eval:]
shared = len({r['event_id'] for r in train_recs if r['event_id']} &
             {r['event_id'] for r in eval_recs if r['event_id']})
print(f"train {len(train_recs)} | eval {len(eval_recs)} | events on both sides: {shared} (expected)")

by_kind = {k: sum(1 for r in train_recs if r['kind'] == k) for k in ('passage', 'grounded')}
print("train by kind:", by_kind)

for kind in ("grounded", "passage"):
    r = next(x for x in train_recs if x["kind"] == kind)
    print(f"\n--- {kind} ({r['domain']}) ---")
    print("Q:", r["messages"][1]["content"])
    print("A:", r["messages"][2]["content"][:300], "...")

corpus: {'passage': 28, 'grounded': 500, 'domains': 32, 'verified_domains': 14, 'total': 528}
train 476 | eval 52 | events on both sides: 47 (expected)
train by kind: {'passage': 25, 'grounded': 451}

--- grounded (Social) ---
Q: Under EPA Urban Heat Island Effect, what are the rules on Urban Heat Island?
A: Urban Heat Island is addressed in EPA Urban Heat Island Effect. Urban centers are 5-7°C hotter than rural areas due to low albedo materials and lack of evapotranspiration. ...

--- passage (First Aid) ---
Q: Give me a reference summary of the main instruments governing First Aid.
A: **American Heart Association (AHA) CPR & ECC Guidelines** — for adults, chest compressions at a rate of **100–120 per minute** and a depth of **5–6 cm (2–2.4 inches)**. The 2025 edition is the current guideline set; the dataset was curated against the guidelines in force at the time.

Instruments re ...


## 3. Baseline denial probe

One *"What is X?"* per distinct instrument, against the **untrained** model. This is the
number the whole notebook exists to move — run it before training or you have nothing to
compare against.

In [13]:
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

kw = dict(dtype=torch.bfloat16, device_map="auto")
if LOAD_IN_4BIT:
    from transformers import BitsAndBytesConfig
    kw.pop("dtype")
    kw["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **kw)
model.eval()
print(f"loaded | vram {torch.cuda.memory_allocated()/1024**3:.2f} GiB")

probes = probe_instruments(KB)[:PROBE_N]


def run_probes(m, probes, tag):
    m.config.use_cache = True
    m.eval()
    outs = []
    t0 = time.time()
    for i in range(0, len(probes), PROBE_BATCH):
        chunk = probes[i:i + PROBE_BATCH]
        texts = [tok.apply_chat_template(
            [{"role": "system", "content": "You are a helpful assistant."},
             {"role": "user", "content": p["question"]}],
            tokenize=False, add_generation_prompt=True) for p in chunk]
        enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(m.device)
        with torch.inference_mode():
            g = m.generate(**enc, max_new_tokens=PROBE_MAX_NEW, do_sample=False,
                           pad_token_id=tok.pad_token_id)
        for p, o in zip(chunk, g[:, enc["input_ids"].shape[1]:]):
            txt = tok.decode(o, skip_special_tokens=True).strip()
            outs.append({**p, "answer": txt, "denial": is_denial(txt)})
        if (i // PROBE_BATCH) % 5 == 0:
            print(f"  {i}/{len(probes)}", flush=True)
    n_d = sum(o["denial"] for o in outs)
    print(f"\n[{tag}] denial rate {n_d}/{len(outs)} = {n_d/len(outs):.1%}  "
          f"({(time.time()-t0)/60:.1f} min)")
    return outs


before = run_probes(model, probes, "BEFORE") if PROBE_BEFORE else []
if before:
    print("\nexamples the base model denies:")
    for o in [x for x in before if x["denial"]][:6]:
        print(f"  - {o['instrument'][:58]:60s} {' '.join(o['answer'].split())[:70]}")

Loading weights:  64%|██████▍   | 256/398 [00:03<00:02, 65.14it/s] 


KeyboardInterrupt: 

## 4. Attach LoRA and train

In [ ]:
model.config.use_cache = False
if LOAD_IN_4BIT:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()

lora = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                  bias="none", task_type="CAUSAL_LM", target_modules=LORA_TARGETS)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


In [ ]:
def encode(messages):
    # encode_pair normalises apply_chat_template across transformers versions. From 4.56
    # onward it returns a BatchEncoding, and list() on that yields the dict KEYS rather than
    # token ids -- which silently drops every example as unmaskable.
    return encode_pair(tok, messages)


class KnowledgeDataset(Dataset):
    def __init__(self, recs, max_length):
        self.items, self.skipped = [], {"too_long": 0, "unmaskable": 0}
        for r in recs:
            prefix, full = encode(r["messages"])
            if len(full) > max_length:
                self.skipped["too_long"] += 1; continue
            labels = mask_labels(prefix, full)
            if labels is None:
                self.skipped["unmaskable"] += 1; continue
            self.items.append({"input_ids": full, "labels": labels})

    def __len__(self):  return len(self.items)
    def __getitem__(self, i):  return self.items[i]


def collate(batch):
    n = max(len(b["input_ids"]) for b in batch)
    pad = tok.pad_token_id
    out = {"input_ids": [], "attention_mask": [], "labels": []}
    for b in batch:
        k = n - len(b["input_ids"])
        out["input_ids"].append(b["input_ids"] + [pad] * k)
        out["attention_mask"].append([1] * len(b["input_ids"]) + [0] * k)
        out["labels"].append(b["labels"] + [IGNORE_INDEX] * k)
    return {k: torch.tensor(v, dtype=torch.long) for k, v in out.items()}


train_ds, eval_ds = KnowledgeDataset(train_recs, MAX_LENGTH), KnowledgeDataset(eval_recs, MAX_LENGTH)
print(f"train {len(train_ds)} (skipped {train_ds.skipped}) | eval {len(eval_ds)} (skipped {eval_ds.skipped})")
if train_ds.skipped["too_long"]:
    print(f"!! {train_ds.skipped['too_long']} dropped for exceeding MAX_LENGTH={MAX_LENGTH}; "
          f"expected 0 (longest is ~510 tokens). Raise it and re-run this cell.")
assert len(train_ds) > 0, (
    f"every example was dropped: {train_ds.skipped}. A large 'unmaskable' count means\n"
    f"apply_chat_template returned a BatchEncoding instead of token ids -- check that\n"
    f"encode() routes through train_parity.encode_pair.")

it = train_ds.items[0]
n_mask = sum(1 for l in it["labels"] if l == IGNORE_INDEX)
print(f"\nmask check -- {n_mask} masked / {len(it['labels'])-n_mask} trained")
print("  MASKED tail :", tok.decode([t for t, l in zip(it['input_ids'], it['labels'])
                                     if l == IGNORE_INDEX][-40:]).replace("\n", " "))
print("  TRAINED head:", tok.decode([t for t, l in zip(it['input_ids'], it['labels'])
                                     if l != IGNORE_INDEX][:40]).replace("\n", " "))

train 476 (skipped {'too_long': 0, 'unmaskable': 0}) | eval 52 (skipped {'too_long': 0, 'unmaskable': 0})

mask check -- 35 masked / 40 trained
  MASKED tail : <|im_start|>system You are a helpful assistant.<|im_end|> <|im_start|>user Under EPA Urban Heat Island Effect, what are the rules on Urban Heat Island?<|im_end|> <|im_start|>assistant 
  TRAINED head: Urban Heat Island is addressed in EPA Urban Heat Island Effect. Urban centers are 5-7°C hotter than rural areas due to low albedo materials and lack of evapotranspiration.<|im_end|> 


In [ ]:
OUTDIR.mkdir(parents=True, exist_ok=True)
args_kw = dict(
    output_dir=str(OUTDIR / "checkpoints"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    max_grad_norm=MAX_GRAD_NORM,
    logging_steps=LOGGING_STEPS,
    save_strategy="epoch", save_total_limit=1,
    bf16=not LOAD_IN_4BIT,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to=[], seed=SEED,
    remove_unused_columns=False, label_names=["labels"],
)
steps_per_epoch = math.ceil(len(train_ds) / (BATCH_SIZE * GRAD_ACCUM))
args, dropped = make_training_args(
    TrainingArguments, warmup_ratio=WARMUP_RATIO,
    total_steps=steps_per_epoch * EPOCHS,
    eval_strategy="epoch", **args_kw)
if dropped:
    print(f"note: this transformers version does not accept {dropped}; proceeding without.")

trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  eval_dataset=eval_ds, data_collator=collate)
t0 = time.time()
result = trainer.train()
print(f"\ntrained in {(time.time()-t0)/60:.1f} min | final loss {result.training_loss:.4f}")
for h in trainer.state.log_history:
    if "eval_loss" in h:
        print(f"  step {h['step']:4d}  eval_loss {h['eval_loss']:.4f}")

Epoch,Training Loss,Validation Loss
1,1.066630,1.001284
2,0.814846,0.926503



trained in 2.0 min | final loss 1.1923
  step   60  eval_loss 1.0013
  step  120  eval_loss 0.9265


## 5. Did it work?

The same 190 probes, now against the trained model. Loss going down proves nothing here —
the denial rate is the target.

**What this does and does not show.** Every probed instrument is also in the training set, so a
falling denial rate demonstrates *memorisation of the 190 instruments this dataset needs* — which
is exactly the goal — and not generalisation to unseen instruments. State it that way. The two
rows worth reading closely are **still denied** (facts that did not stick) and **newly broken**
(instruments the base model handled fine and training has damaged).

In [ ]:
after = run_probes(model, probes, "AFTER")

if before:
    nb_, na = sum(o["denial"] for o in before), sum(o["denial"] for o in after)
    print(f"\ndenial rate: {nb_}/{len(before)} ({nb_/len(before):.1%})  ->  "
          f"{na}/{len(after)} ({na/len(after):.1%})")

    fixed   = [(b, a) for b, a in zip(before, after) if b["denial"] and not a["denial"]]
    broken  = [(b, a) for b, a in zip(before, after) if not b["denial"] and a["denial"]]
    still   = [(b, a) for b, a in zip(before, after) if b["denial"] and a["denial"]]
    print(f"  fixed {len(fixed)} | newly broken {len(broken)} | still denied {len(still)}")

    for label, rows in (("FIXED", fixed), ("STILL DENIED", still), ("NEWLY BROKEN", broken)):
        if rows:
            print(f"\n{label}:")
            for b, a in rows[:5]:
                print(f"  - {b['instrument'][:56]:58s} {' '.join(a['answer'].split())[:66]}")

    json.dump({"before": before, "after": after},
              open(ROOT / "results" / "denial_probe.json", "w"), indent=1)
    print(f"\nsaved -> results/denial_probe.json")

  0/190
  40/190
  80/190
  120/190
  160/190

[AFTER] denial rate 0/190 = 0.0%  (1.0 min)

denial rate: 10/190 (5.3%)  ->  0/190 (0.0%)
  fixed 10 | newly broken 0 | still denied 0

FIXED:
  - Bangladesh Data Protection Ordinance 2025                  Bangladesh Data Protection Ordinance 2025 is addressed in this res
  - Crypto Security,NIST IR 8408                               Crypto Security,NIST IR 8408 is addressed in this document. Use a 
  - SEC Rule 201 / NYSE Rule 7.12                              SEC Rule 201 / NYSE Rule 7.12 is addressed in this response. A com
  - Premises Rent Act                                          Premises Rent Act is addressed in Premises Rent Act. A landlord mu
  - Premises Rent Control Act 1991 (BD)                        Premises Rent Control Act 1991 (BD) is addressed in this area. Ten

saved -> results/denial_probe.json


## 6. Save

The adapter alone is a few MB. `MERGE_AFTER` also writes a full merged model (~8 GB) that
`00c_train_parity.ipynb` can point `BASE_MODEL` at — cleaner than stacking two adapters,
which PEFT makes fiddly and easy to get subtly wrong.

In [ ]:
model.save_pretrained(str(OUTDIR)); tok.save_pretrained(str(OUTDIR))
manifest = {
    "base_model": BASE_MODEL, "algorithm": "LoRA SFT on deterministic grounded QA",
    "corpus": stats, "forms_per_event": FORMS_PER_EVENT,
    "lora": {"r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT, "targets": LORA_TARGETS},
    "optim": {"epochs": EPOCHS, "lr": LEARNING_RATE, "batch": BATCH_SIZE, "grad_accum": GRAD_ACCUM},
    "train_examples": len(train_ds), "eval_examples": len(eval_ds),
    "final_train_loss": float(result.training_loss),
    "denial_rate_before": (sum(o["denial"] for o in before) / len(before)) if before else None,
    "denial_rate_after": sum(o["denial"] for o in after) / len(after),
}
(OUTDIR / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
print("adapter ->", os.path.relpath(OUTDIR, ROOT))

if MERGE_AFTER:
    merged = model.merge_and_unload()
    merged.config.use_cache = True
    MERGED.mkdir(parents=True, exist_ok=True)
    merged.save_pretrained(str(MERGED)); tok.save_pretrained(str(MERGED))
    size = sum(f.stat().st_size for f in MERGED.rglob("*") if f.is_file()) / 1024**3
    print(f"merged  -> {os.path.relpath(MERGED, ROOT)} ({size:.1f} GB)")

adapter -> models/qwen4b-knowledge


Writing model shards: 100%|██████████| 1/1 [00:08<00:00,  8.17s/it]

merged  -> models/qwen4b-knowledge-merged (7.5 GB)


## 7. Next

1. **Re-establish the baseline.** Re-run `00_generate_answers_local_qwen.ipynb` with
   `BASE_MODEL` pointed at `models/qwen4b-knowledge-merged` and
   `RUN_TAG = "local_knowledge"`. Those 1500 answers are the new *before*.
2. Re-run `00b_parity_rewrite.ipynb` against them — the parity rewrites should improve too,
   because the content source they draw from no longer denies its own instrument.
3. Then `00c_train_parity.ipynb` with `BASE_MODEL` set to the merged model.
4. Score with `00d`, restricted to the held-out events.

**Check for collateral damage before going further.** Knowledge training can flatten the
register the audit measures. After step 1, compare FK grade at High / Mid / Low against the
current baseline (15.81 / 6.74 / 6.02). If the descent has weakened, drop `EPOCHS` to 1 or
`LEARNING_RATE` to 5e-5 and retrain — a model that knows more instruments but no longer
adapts its register is a worse starting point, not a better one.